In [ ]:
from snowflake.snowpark import Session
session = Session.builder.configs({
    "account": "CQOUAQZ-REVEEL_AZURE", 
    "user":"RAMYA-SQUADRON@REVEELGROUP.COM", # e.g. xy12345.ca-central-1
    "warehouse": "DEV_WH",
    "database": "STAGING",
    "schema": "AUDIT",
    "role": "SYSADMIN",
    "authenticator": "externalbrowser",
    "client_session_keep_alive": True
}).create()


import sys
sys.path.append("C:/Users/nramy/OneDrive/Documents/Reveel/Snowflake")

In [3]:
# Snowflake notebook source
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

from snowflake.snowpark import DataFrame, Session
from snowflake.snowpark.functions import col

In [4]:
from notebooks import sfutils
from notebooks.utils.edi_schemas import FEDEX_RAW_EDI_JSON_SCHEMA
from notebooks.utils.table_names import (
    FEDEX_EDI_110_4010_RAW,
    FEDEX_EDI_110_4060_RAW,
    FEDEX_EDI_210_RAW,
)
from notebooks.etl.widgets import catalog

from notebooks.misc_utils import read_files_with_filename

In [5]:
sfutils.widgets.text("days_before", "-1") 
sfutils.widgets.text("overwrite_schema", "true") 

'true'

In [6]:
days_before = sfutils.widgets.get("days_before")
overwrite_schema = sfutils.widgets.get("overwrite_schema")

In [7]:
#days_before = -1
days_before = int(days_before)
print(f"days_before: {days_before}")

days_before: -1


In [8]:
print(days_before, overwrite_schema, catalog)

-1 true staging


In [9]:
azure_url = {
    f"@{catalog}.public.sf_stage_invoice" : "azure://stpreproccarrinvprdwu01.blob.core.windows.net/maincontainer/",
    f"@{catalog}.public.sf_stage_edi" : "azure://stinbdataediprdwu01.blob.core.windows.net/maincontainer/",
    f"@{catalog}.public.sf_stage" : "azure://stcarrshiptrckapiprdwu01.blob.core.windows.net/maincontainer/"
}

In [10]:

sfutils.widgets.text("stage_name", "sf_stage_edi")
stage_name = sfutils.widgets.get("stage_name")

stage_suffix = f"@{catalog}.public.{stage_name}"
azure_url_path = azure_url[stage_suffix]

print(overwrite_schema, stage_suffix, azure_url_path)

true @staging.public.sf_stage_edi azure://stinbdataediprdwu01.blob.core.windows.net/maincontainer/


In [ ]:
stage = "STAGING.PUBLIC.SF_STAGE_EDI"
carrier = "fedex"
entire_path = f"{stage}/carrier={carrier}/"
location = f"carrier={carrier}"

In [ ]:
# Build final SQL 

file_format="test_ndjson_format"
stage_name = entire_path

sql_query = f"""
    SELECT  
        PARSE_JSON($1):delimiters::varchar as "delimiters",
        PARSE_JSON($1):envelope::varchar as "envelope",
        PARSE_JSON($1:transactionSets)[0]::variant as "transactionSets",
        PARSE_JSON($1):upload_year::NUMBER(38,0) as "upload_year",
        PARSE_JSON($1):upload_month::NUMBER(38,0) as "upload_month",
        PARSE_JSON($1):upload_day::NUMBER(38,0) as "upload_day",
        PARSE_JSON($1:transactionSets)[0].heading.transaction_set_header_ST.transaction_set_identifier_code_01::string as "edi_type",
        CONCAT('{azure_url_path}', '/', METADATA$FILENAME) as "file_path"
    FROM @{stage_name}
    (FILE_FORMAT => '{file_format}')
"""
print(f"Executing query against stage: {stage_name}")
df = session.sql(sql_query)

In [14]:
df.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"delimiters"                                        |"envelope"                                          |"transactionSets"                                   |"upload_year"  |"upload_month"  |"upload_day"  |"edi_type"  |"file_path"                                         |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|{"composite":"\\","element":"*","repetition":"^...  |{"groupHeader":{"agencyCode":"X","applicationRe...  |{                                                   |2023           

In [15]:
df.print_schema()

root
 |-- "delimiters": StringType() (nullable = True)
 |-- "envelope": StringType() (nullable = True)
 |-- "transactionSets": VariantType() (nullable = True)
 |-- "upload_year": LongType() (nullable = True)
 |-- "upload_month": LongType() (nullable = True)
 |-- "upload_day": LongType() (nullable = True)
 |-- "edi_type": StringType() (nullable = True)
 |-- "file_path": StringType(16777281) (nullable = True)


In [16]:
raw_110_4060 = df.where(((F.col('"edi_type"') == "110"))  
                         & (F.parse_json(col('"envelope"'))["interchangeHeader"]["controlVersionNumber"] == "00406"))

In [17]:
raw_110_4010 = df.where(((F.col('"edi_type"') == "110"))
                         & (F.parse_json(col('"envelope"'))["interchangeHeader"]["controlVersionNumber"] == "00401"))

In [18]:
raw_210 = df.where(F.col('"edi_type"') == "210")

In [19]:
raw_110_4060.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"delimiters"                                        |"envelope"                                          |"transactionSets"                                   |"upload_year"  |"upload_month"  |"upload_day"  |"edi_type"  |"file_path"                                         |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|{"composite":"\\","element":"*","repetition":"^...  |{"groupHeader":{"agencyCode":"X","applicationRe...  |{                                                   |2023           

In [20]:
df1=df.filter(F.col('"upload_year"') == 2023).limit(100)


In [21]:
df1 = df.filter(F.col('"upload_year"').isNotNull())

In [22]:
df1.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"delimiters"                                        |"envelope"                                          |"transactionSets"                                   |"upload_year"  |"upload_month"  |"upload_day"  |"edi_type"  |"file_path"                                         |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|{"composite":"\\","element":"*","repetition":"^...  |{"groupHeader":{"agencyCode":"X","applicationRe...  |{                                                   |2023           